In [1]:
import os
import time
import requests
import pandas as pd
import sqlite3
from datetime import datetime, timedelta

API_KEY = "4ddf855d7f224af395ce8ed58f80babd"
TICKERS = ['AAPL', 'MSFT', 'GOOGL']
COMPANY_NAMES = {
    'AAPL': 'Apple',
    'MSFT': 'Microsoft',
    'GOOGL': 'Google OR Alphabet'
}
DAYS_BACK = 29
BLOCK_DAYS = 5
MIN_BLOCK = 1
PAGE_SIZE = 100

NEWS_DIR = "data/news"
os.makedirs(NEWS_DIR, exist_ok=True)
DB_PATH = f"{NEWS_DIR}/tech_news.db"
URL = "https://newsapi.org/v2/everything"

def adaptive_fetch(api_url, api_key, query, ticker, start_date, end_date, block_days=BLOCK_DAYS, min_block=MIN_BLOCK, verbose=True):
    all_articles = []
    dt_start = datetime.strptime(start_date, "%Y-%m-%d")
    dt_end   = datetime.strptime(end_date, "%Y-%m-%d")
    while dt_start < dt_end:
        dt_block_end = min(dt_end, dt_start + timedelta(days=block_days))
        params = {
            "q": query,
            "from": dt_start.strftime("%Y-%m-%d"),
            "to": dt_block_end.strftime("%Y-%m-%d"),
            "language": "en",
            "sortBy": "publishedAt",
            "apiKey": api_key,
            "pageSize": PAGE_SIZE,
            "page": 1,
        }
        if verbose:
            print(f"  {ticker}: Fetching news from {params['from']} to {params['to']}")
        r = requests.get(api_url, params=params)
        if r.status_code != 200:
            print(f"  Error for {ticker}: {r.text}")
            break
        data = r.json()
        n_articles = len(data.get("articles", []))
        if n_articles == PAGE_SIZE and block_days > min_block:
            if verbose:
                print(f"  {ticker}: Splitting dense window {dt_start} to {dt_block_end}")
            mid = dt_start + timedelta(days=block_days // 2)
            all_articles += adaptive_fetch(
                api_url, api_key, query, ticker,
                dt_start.strftime("%Y-%m-%d"), mid.strftime("%Y-%m-%d"),
                block_days=block_days // 2, min_block=min_block, verbose=verbose
            )
            all_articles += adaptive_fetch(
                api_url, api_key, query, ticker,
                mid.strftime("%Y-%m-%d"), dt_block_end.strftime("%Y-%m-%d"),
                block_days=block_days - block_days // 2, min_block=min_block, verbose=verbose
            )
        else:
            for article in data.get("articles", []):
                all_articles.append({
                    "ticker": ticker,
                    "headline": article["title"],
                    "description": article.get("description"),
                    "published": article["publishedAt"],
                    "source": article["source"]["name"],
                    "url": article["url"]
                })
        time.sleep(1.2)
        dt_start = dt_block_end
    return all_articles

all_news = []
date_end = datetime.now()
date_start = date_end - timedelta(days=DAYS_BACK)
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute("""
CREATE TABLE IF NOT EXISTS news (
    ticker TEXT,
    headline TEXT,
    description TEXT,
    published TEXT,
    source TEXT,
    url TEXT,
    PRIMARY KEY (ticker, headline, published)
)
""")
conn.commit()

for ticker in TICKERS:
    query = COMPANY_NAMES[ticker] + f" OR {ticker}"
    print(f"\nFetching news for {ticker} ({query})")
    articles = adaptive_fetch(
        URL, API_KEY, query, ticker,
        date_start.strftime("%Y-%m-%d"), date_end.strftime("%Y-%m-%d"),
        block_days=BLOCK_DAYS, min_block=MIN_BLOCK, verbose=True
    )
    df = pd.DataFrame(articles)
    if not df.empty:
        df = df.drop_duplicates(subset=["headline", "published"])
        df["published"] = pd.to_datetime(df["published"])
        df = df.sort_values("published").reset_index(drop=True)
        df.to_csv(f"{NEWS_DIR}/{ticker}_news.csv", index=False)
        df.to_sql("news", conn, if_exists="append", index=False, method="multi")
        print(f"  {len(df)} articles saved to {NEWS_DIR}/{ticker}_news.csv and database.")
        all_news.append(df)
    else:
        print(f"  No news for {ticker} in the past {DAYS_BACK} days.")

if all_news:
    news_df = pd.concat(all_news, ignore_index=True)
    news_df = news_df.drop_duplicates(subset=["ticker", "headline", "published"])
    news_df = news_df.sort_values(["ticker", "published"]).reset_index(drop=True)
    news_df.to_csv(f"{NEWS_DIR}/tech_news.csv", index=False)
    print(f"\nTotal unique articles collected: {news_df.shape[0]}")
    print(f"Saved combined news to {NEWS_DIR}/tech_news.csv")
    print(f"All news also stored in SQLite DB: {DB_PATH}")
    print(news_df.head())
else:
    print("No news found for any ticker.")

conn.close()


Fetching news for AAPL (Apple OR AAPL)
  AAPL: Fetching news from 2025-06-12 to 2025-06-17
  AAPL: Fetching news from 2025-06-17 to 2025-06-22
  AAPL: Fetching news from 2025-06-22 to 2025-06-27
  AAPL: Splitting dense window 2025-06-22 00:00:00 to 2025-06-27 00:00:00
  AAPL: Fetching news from 2025-06-22 to 2025-06-24
  AAPL: Fetching news from 2025-06-24 to 2025-06-27
  AAPL: Splitting dense window 2025-06-24 00:00:00 to 2025-06-27 00:00:00
  AAPL: Fetching news from 2025-06-24 to 2025-06-25
  AAPL: Fetching news from 2025-06-25 to 2025-06-27
  AAPL: Splitting dense window 2025-06-25 00:00:00 to 2025-06-27 00:00:00
  AAPL: Fetching news from 2025-06-25 to 2025-06-26
  AAPL: Fetching news from 2025-06-26 to 2025-06-27
  AAPL: Fetching news from 2025-06-27 to 2025-07-02
  AAPL: Fetching news from 2025-07-02 to 2025-07-07
  AAPL: Fetching news from 2025-07-07 to 2025-07-11
  874 articles saved to data/news/AAPL_news.csv and database.

Fetching news for MSFT (Microsoft OR MSFT)
  MSFT: 

In [2]:
import os
import time
import requests
import pandas as pd
import sqlite3
from datetime import datetime, timedelta

API_KEY = "4ddf855d7f224af395ce8ed58f80babd"
TICKERS = ['AMZN', 'META', 'NVDA']
COMPANY_NAMES = {
    'AMZN': 'Amazon',
    'META': 'Meta OR Facebook',
    'NVDA': 'Nvidia'
}
DAYS_BACK = 29
BLOCK_DAYS = 5
MIN_BLOCK = 1
PAGE_SIZE = 100

NEWS_DIR = "data/news"
os.makedirs(NEWS_DIR, exist_ok=True)
DB_PATH = f"{NEWS_DIR}/tech_news.db"
URL = "https://newsapi.org/v2/everything"

def adaptive_fetch(api_url, api_key, query, ticker, start_date, end_date, block_days=BLOCK_DAYS, min_block=MIN_BLOCK, verbose=True):
    all_articles = []
    dt_start = datetime.strptime(start_date, "%Y-%m-%d")
    dt_end   = datetime.strptime(end_date, "%Y-%m-%d")
    while dt_start < dt_end:
        dt_block_end = min(dt_end, dt_start + timedelta(days=block_days))
        params = {
            "q": query,
            "from": dt_start.strftime("%Y-%m-%d"),
            "to": dt_block_end.strftime("%Y-%m-%d"),
            "language": "en",
            "sortBy": "publishedAt",
            "apiKey": api_key,
            "pageSize": PAGE_SIZE,
            "page": 1,
        }
        if verbose:
            print(f"  {ticker}: Fetching news from {params['from']} to {params['to']}")
        r = requests.get(api_url, params=params)
        if r.status_code != 200:
            print(f"  Error for {ticker}: {r.text}")
            break
        data = r.json()
        n_articles = len(data.get("articles", []))
        if n_articles == PAGE_SIZE and block_days > min_block:
            if verbose:
                print(f"  {ticker}: Splitting dense window {dt_start} to {dt_block_end}")
            mid = dt_start + timedelta(days=block_days // 2)
            all_articles += adaptive_fetch(
                api_url, api_key, query, ticker,
                dt_start.strftime("%Y-%m-%d"), mid.strftime("%Y-%m-%d"),
                block_days=block_days // 2, min_block=min_block, verbose=verbose
            )
            all_articles += adaptive_fetch(
                api_url, api_key, query, ticker,
                mid.strftime("%Y-%m-%d"), dt_block_end.strftime("%Y-%m-%d"),
                block_days=block_days - block_days // 2, min_block=min_block, verbose=verbose
            )
        else:
            for article in data.get("articles", []):
                all_articles.append({
                    "ticker": ticker,
                    "headline": article["title"],
                    "description": article.get("description"),
                    "published": article["publishedAt"],
                    "source": article["source"]["name"],
                    "url": article["url"]
                })
        time.sleep(1.2)
        dt_start = dt_block_end
    return all_articles

all_news = []
date_end = datetime.now()
date_start = date_end - timedelta(days=DAYS_BACK)

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute("""
CREATE TABLE IF NOT EXISTS news (
    ticker TEXT,
    headline TEXT,
    description TEXT,
    published TEXT,
    source TEXT,
    url TEXT,
    PRIMARY KEY (ticker, headline, published)
)
""")
conn.commit()

for ticker in TICKERS:
    query = COMPANY_NAMES[ticker] + f" OR {ticker}"
    print(f"\nFetching news for {ticker} ({query})")
    articles = adaptive_fetch(
        URL, API_KEY, query, ticker,
        date_start.strftime("%Y-%m-%d"), date_end.strftime("%Y-%m-%d"),
        block_days=BLOCK_DAYS, min_block=MIN_BLOCK, verbose=True
    )
    df = pd.DataFrame(articles)
    if not df.empty:
        df = df.drop_duplicates(subset=["headline", "published"])
        df["published"] = pd.to_datetime(df["published"])
        df = df.sort_values("published").reset_index(drop=True)
        df.to_csv(f"{NEWS_DIR}/{ticker}_news.csv", index=False)
        df.to_sql("news", conn, if_exists="append", index=False, method="multi")
        print(f"  {len(df)} articles saved to {NEWS_DIR}/{ticker}_news.csv and database.")
        all_news.append(df)
    else:
        print(f"  No news for {ticker} in the past {DAYS_BACK} days.")

if all_news:
    news_df = pd.concat(all_news, ignore_index=True)
    news_df = news_df.drop_duplicates(subset=["ticker", "headline", "published"])
    news_df = news_df.sort_values(["ticker", "published"]).reset_index(drop=True)
    news_df.to_csv(f"{NEWS_DIR}/tech_news.csv", index=False)
    print(f"\nTotal unique articles collected: {news_df.shape[0]}")
    print(f"Saved combined news to {NEWS_DIR}/tech_news.csv")
    print(f"All news also stored in SQLite DB: {DB_PATH}")
    print(news_df.head())
else:
    print("No news found for any ticker.")

conn.close()


Fetching news for AMZN (Amazon OR AMZN)
  AMZN: Fetching news from 2025-06-12 to 2025-06-17
  AMZN: Splitting dense window 2025-06-12 00:00:00 to 2025-06-17 00:00:00
  AMZN: Fetching news from 2025-06-12 to 2025-06-14
  AMZN: Splitting dense window 2025-06-12 00:00:00 to 2025-06-14 00:00:00
  AMZN: Fetching news from 2025-06-12 to 2025-06-13
  AMZN: Fetching news from 2025-06-13 to 2025-06-14
  AMZN: Fetching news from 2025-06-14 to 2025-06-17
  AMZN: Splitting dense window 2025-06-14 00:00:00 to 2025-06-17 00:00:00
  AMZN: Fetching news from 2025-06-14 to 2025-06-15
  AMZN: Fetching news from 2025-06-15 to 2025-06-17
  AMZN: Splitting dense window 2025-06-15 00:00:00 to 2025-06-17 00:00:00
  AMZN: Fetching news from 2025-06-15 to 2025-06-16
  AMZN: Fetching news from 2025-06-16 to 2025-06-17
  AMZN: Fetching news from 2025-06-17 to 2025-06-22
  AMZN: Fetching news from 2025-06-22 to 2025-06-27
  AMZN: Fetching news from 2025-06-27 to 2025-07-02
  AMZN: Fetching news from 2025-07-02 t

In [6]:
import os
import time
import requests
import pandas as pd
import sqlite3
from datetime import datetime, timedelta

API_KEY = "4ddf855d7f224af395ce8ed58f80babd"
TICKERS = ['TSLA', 'ORCL']
COMPANY_NAMES = {
    'TSLA': 'Tesla',
    'ORCL': 'Oracle'
}
DAYS_BACK = 29
BLOCK_DAYS = 5
MIN_BLOCK = 1
PAGE_SIZE = 100

NEWS_DIR = "data/news"
os.makedirs(NEWS_DIR, exist_ok=True)
DB_PATH = f"{NEWS_DIR}/tech_news.db"
URL = "https://newsapi.org/v2/everything"

def adaptive_fetch(api_url, api_key, query, ticker, start_date, end_date, block_days=BLOCK_DAYS, min_block=MIN_BLOCK, verbose=True):
    all_articles = []
    dt_start = datetime.strptime(start_date, "%Y-%m-%d")
    dt_end   = datetime.strptime(end_date, "%Y-%m-%d")
    while dt_start < dt_end:
        dt_block_end = min(dt_end, dt_start + timedelta(days=block_days))
        params = {
            "q": query,
            "from": dt_start.strftime("%Y-%m-%d"),
            "to": dt_block_end.strftime("%Y-%m-%d"),
            "language": "en",
            "sortBy": "publishedAt",
            "apiKey": api_key,
            "pageSize": PAGE_SIZE,
            "page": 1,
        }
        if verbose:
            print(f"  {ticker}: Fetching news from {params['from']} to {params['to']}")
        r = requests.get(api_url, params=params)
        if r.status_code != 200:
            print(f"  Error for {ticker}: {r.text}")
            break
        data = r.json()
        n_articles = len(data.get("articles", []))
        if n_articles == PAGE_SIZE and block_days > min_block:
            if verbose:
                print(f"  {ticker}: Splitting dense window {dt_start} to {dt_block_end}")
            mid = dt_start + timedelta(days=block_days // 2)
            all_articles += adaptive_fetch(
                api_url, api_key, query, ticker,
                dt_start.strftime("%Y-%m-%d"), mid.strftime("%Y-%m-%d"),
                block_days=block_days // 2, min_block=min_block, verbose=verbose
            )
            all_articles += adaptive_fetch(
                api_url, api_key, query, ticker,
                mid.strftime("%Y-%m-%d"), dt_block_end.strftime("%Y-%m-%d"),
                block_days=block_days - block_days // 2, min_block=min_block, verbose=verbose
            )
        else:
            for article in data.get("articles", []):
                all_articles.append({
                    "ticker": ticker,
                    "headline": article["title"],
                    "description": article.get("description"),
                    "published": article["publishedAt"],
                    "source": article["source"]["name"],
                    "url": article["url"]
                })
        time.sleep(1.2)
        dt_start = dt_block_end
    return all_articles

all_news = []
date_end = datetime.now()
date_start = date_end - timedelta(days=DAYS_BACK)
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute("""
CREATE TABLE IF NOT EXISTS news (
    ticker TEXT,
    headline TEXT,
    description TEXT,
    published TEXT,
    source TEXT,
    url TEXT,
    PRIMARY KEY (ticker, headline, published)
)
""")
conn.commit()

for ticker in TICKERS:
    query = COMPANY_NAMES[ticker] + f" OR {ticker}"
    print(f"\nFetching news for {ticker} ({query})")
    articles = adaptive_fetch(
        URL, API_KEY, query, ticker,
        date_start.strftime("%Y-%m-%d"), date_end.strftime("%Y-%m-%d"),
        block_days=BLOCK_DAYS, min_block=MIN_BLOCK, verbose=True
    )
    df = pd.DataFrame(articles)
    if not df.empty:
        df = df.drop_duplicates(subset=["headline", "published"])
        df["published"] = pd.to_datetime(df["published"])
        df = df.sort_values("published").reset_index(drop=True)
        df.to_csv(f"{NEWS_DIR}/{ticker}_news.csv", index=False)
        df.to_sql("news", conn, if_exists="append", index=False, method="multi")
        print(f"  {len(df)} articles saved to {NEWS_DIR}/{ticker}_news.csv and database.")
        all_news.append(df)
    else:
        print(f"  No news for {ticker} in the past {DAYS_BACK} days.")

if all_news:
    news_df = pd.concat(all_news, ignore_index=True)
    news_df = news_df.drop_duplicates(subset=["ticker", "headline", "published"])
    news_df = news_df.sort_values(["ticker", "published"]).reset_index(drop=True)
    news_df.to_csv(f"{NEWS_DIR}/tech_news.csv", index=False)
    print(f"\nTotal unique articles collected: {news_df.shape[0]}")
    print(f"Saved combined news to {NEWS_DIR}/tech_news.csv")
    print(f"All news also stored in SQLite DB: {DB_PATH}")
    print(news_df.head())
else:
    print("No news found for any ticker.")

conn.close()


Fetching news for TSLA (Tesla OR TSLA)
  TSLA: Fetching news from 2025-06-12 to 2025-06-17
  TSLA: Fetching news from 2025-06-17 to 2025-06-22
  TSLA: Splitting dense window 2025-06-17 00:00:00 to 2025-06-22 00:00:00
  TSLA: Fetching news from 2025-06-17 to 2025-06-19
  TSLA: Fetching news from 2025-06-19 to 2025-06-22
  TSLA: Splitting dense window 2025-06-19 00:00:00 to 2025-06-22 00:00:00
  TSLA: Fetching news from 2025-06-19 to 2025-06-20
  TSLA: Fetching news from 2025-06-20 to 2025-06-22
  TSLA: Splitting dense window 2025-06-20 00:00:00 to 2025-06-22 00:00:00
  TSLA: Fetching news from 2025-06-20 to 2025-06-21
  TSLA: Fetching news from 2025-06-21 to 2025-06-22
  TSLA: Fetching news from 2025-06-22 to 2025-06-27
  TSLA: Fetching news from 2025-06-27 to 2025-07-02
  TSLA: Fetching news from 2025-07-02 to 2025-07-07
  TSLA: Fetching news from 2025-07-07 to 2025-07-11
  TSLA: Splitting dense window 2025-07-07 00:00:00 to 2025-07-11 00:00:00
  TSLA: Fetching news from 2025-07-07 to

In [7]:
import os
import time
import requests
import pandas as pd
import sqlite3
from datetime import datetime, timedelta

API_KEY = "b4d783efd03e4983ad0b2eee085cbbaa"
TICKERS = ['IBM', 'CRM']
COMPANY_NAMES = {
    'IBM': 'IBM',
    'CRM': 'Salesforce'
}
DAYS_BACK = 29
BLOCK_DAYS = 5
MIN_BLOCK = 1
PAGE_SIZE = 100

NEWS_DIR = "data/news"
os.makedirs(NEWS_DIR, exist_ok=True)
DB_PATH = f"{NEWS_DIR}/tech_news.db"
URL = "https://newsapi.org/v2/everything"

def adaptive_fetch(api_url, api_key, query, ticker, start_date, end_date, block_days=BLOCK_DAYS, min_block=MIN_BLOCK, verbose=True):
    all_articles = []
    dt_start = datetime.strptime(start_date, "%Y-%m-%d")
    dt_end   = datetime.strptime(end_date, "%Y-%m-%d")
    while dt_start < dt_end:
        dt_block_end = min(dt_end, dt_start + timedelta(days=block_days))
        params = {
            "q": query,
            "from": dt_start.strftime("%Y-%m-%d"),
            "to": dt_block_end.strftime("%Y-%m-%d"),
            "language": "en",
            "sortBy": "publishedAt",
            "apiKey": api_key,
            "pageSize": PAGE_SIZE,
            "page": 1,
        }
        if verbose:
            print(f"  {ticker}: Fetching news from {params['from']} to {params['to']}")
        r = requests.get(api_url, params=params)
        if r.status_code != 200:
            print(f"  Error for {ticker}: {r.text}")
            break
        data = r.json()
        n_articles = len(data.get("articles", []))
        if n_articles == PAGE_SIZE and block_days > min_block:
            if verbose:
                print(f"  {ticker}: Splitting dense window {dt_start} to {dt_block_end}")
            mid = dt_start + timedelta(days=block_days // 2)
            all_articles += adaptive_fetch(
                api_url, api_key, query, ticker,
                dt_start.strftime("%Y-%m-%d"), mid.strftime("%Y-%m-%d"),
                block_days=block_days // 2, min_block=min_block, verbose=verbose
            )
            all_articles += adaptive_fetch(
                api_url, api_key, query, ticker,
                mid.strftime("%Y-%m-%d"), dt_block_end.strftime("%Y-%m-%d"),
                block_days=block_days - block_days // 2, min_block=min_block, verbose=verbose
            )
        else:
            for article in data.get("articles", []):
                all_articles.append({
                    "ticker": ticker,
                    "headline": article["title"],
                    "description": article.get("description"),
                    "published": article["publishedAt"],
                    "source": article["source"]["name"],
                    "url": article["url"]
                })
        time.sleep(1.2)
        dt_start = dt_block_end
    return all_articles

all_news = []
date_end = datetime.now()
date_start = date_end - timedelta(days=DAYS_BACK)
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute("""
CREATE TABLE IF NOT EXISTS news (
    ticker TEXT,
    headline TEXT,
    description TEXT,
    published TEXT,
    source TEXT,
    url TEXT,
    PRIMARY KEY (ticker, headline, published)
)
""")
conn.commit()

for ticker in TICKERS:
    query = COMPANY_NAMES[ticker] + f" OR {ticker}"
    print(f"\nFetching news for {ticker} ({query})")
    articles = adaptive_fetch(
        URL, API_KEY, query, ticker,
        date_start.strftime("%Y-%m-%d"), date_end.strftime("%Y-%m-%d"),
        block_days=BLOCK_DAYS, min_block=MIN_BLOCK, verbose=True
    )
    df = pd.DataFrame(articles)
    if not df.empty:
        df = df.drop_duplicates(subset=["headline", "published"])
        df["published"] = pd.to_datetime(df["published"])
        df = df.sort_values("published").reset_index(drop=True)
        df.to_csv(f"{NEWS_DIR}/{ticker}_news.csv", index=False)
        df.to_sql("news", conn, if_exists="append", index=False, method="multi")
        print(f"  {len(df)} articles saved to {NEWS_DIR}/{ticker}_news.csv and database.")
        all_news.append(df)
    else:
        print(f"  No news for {ticker} in the past {DAYS_BACK} days.")

if all_news:
    news_df = pd.concat(all_news, ignore_index=True)
    news_df = news_df.drop_duplicates(subset=["ticker", "headline", "published"])
    news_df = news_df.sort_values(["ticker", "published"]).reset_index(drop=True)
    news_df.to_csv(f"{NEWS_DIR}/tech_news.csv", index=False)
    print(f"\nTotal unique articles collected: {news_df.shape[0]}")
    print(f"Saved combined news to {NEWS_DIR}/tech_news.csv")
    print(f"All news also stored in SQLite DB: {DB_PATH}")
    print(news_df.head())
else:
    print("No news found for any ticker.")

conn.close()


Fetching news for IBM (IBM OR IBM)
  IBM: Fetching news from 2025-06-12 to 2025-06-17
  IBM: Splitting dense window 2025-06-12 00:00:00 to 2025-06-17 00:00:00
  IBM: Fetching news from 2025-06-12 to 2025-06-14
  IBM: Splitting dense window 2025-06-12 00:00:00 to 2025-06-14 00:00:00
  IBM: Fetching news from 2025-06-12 to 2025-06-13
  IBM: Fetching news from 2025-06-13 to 2025-06-14
  IBM: Fetching news from 2025-06-14 to 2025-06-17
  IBM: Splitting dense window 2025-06-14 00:00:00 to 2025-06-17 00:00:00
  IBM: Fetching news from 2025-06-14 to 2025-06-15
  IBM: Fetching news from 2025-06-15 to 2025-06-17
  IBM: Splitting dense window 2025-06-15 00:00:00 to 2025-06-17 00:00:00
  IBM: Fetching news from 2025-06-15 to 2025-06-16
  IBM: Fetching news from 2025-06-16 to 2025-06-17
  IBM: Fetching news from 2025-06-17 to 2025-06-22
  IBM: Splitting dense window 2025-06-17 00:00:00 to 2025-06-22 00:00:00
  IBM: Fetching news from 2025-06-17 to 2025-06-19
  IBM: Fetching news from 2025-06-19 t

# API_KEY = "b4d783efd03e4983ad0b2eee085cbbaa" or "4ddf855d7f224af395ce8ed58f80babd"

In [8]:
import sqlite3
import pandas as pd

DB_PATH = "data/news/tech_news.db"
conn = sqlite3.connect(DB_PATH)
query = "SELECT ticker, COUNT(*) as num_articles FROM news GROUP BY ticker"
df = pd.read_sql(query, conn)
num_tickers = df.shape[0]

print(f"Unique tickers in DB: {num_tickers}")
print(df)

conn.close()

Unique tickers in DB: 10
  ticker  num_articles
0   AAPL           874
1   AMZN           976
2    CRM          1227
3  GOOGL          1272
4    IBM           791
5   META           878
6   MSFT           578
7   NVDA          1029
8   ORCL          1167
9   TSLA          1071


In [9]:
import sqlite3
import pandas as pd

DB_PATH = "data/news/tech_news.db"

conn = sqlite3.connect(DB_PATH)

query = """
SELECT
    ticker,
    MIN(published) as earliest,
    MAX(published) as latest,
    COUNT(*) as num_articles
FROM news
GROUP BY ticker
ORDER BY ticker
"""
df = pd.read_sql(query, conn)
df['earliest'] = pd.to_datetime(df['earliest']).dt.strftime('%Y-%m-%d %H:%M')
df['latest'] = pd.to_datetime(df['latest']).dt.strftime('%Y-%m-%d %H:%M')

print(df)

conn.close()


  ticker          earliest            latest  num_articles
0   AAPL  2025-06-17 20:20  2025-07-10 04:07           874
1   AMZN  2025-06-13 20:38  2025-07-10 04:09           976
2    CRM  2025-06-12 12:00  2025-07-10 23:00          1227
3  GOOGL  2025-06-14 17:20  2025-07-10 04:08          1272
4    IBM  2025-06-12 06:35  2025-07-10 18:40           791
5   META  2025-06-17 19:59  2025-07-10 04:06           878
6   MSFT  2025-06-17 19:53  2025-07-10 04:06           578
7   NVDA  2025-06-17 12:47  2025-07-10 23:59          1029
8   ORCL  2025-06-12 22:01  2025-07-10 19:17          1167
9   TSLA  2025-06-17 15:48  2025-07-10 23:36          1071


In [1]:
import pandas as pd
from pathlib import Path

processed_dir = Path("data/processed")
csv_files = list(processed_dir.glob("*_engineered_with_targets.csv"))

for file in csv_files:
    print(f"\n--- {file.name} ---")
    df = pd.read_csv(file)
    print(df.head(), "\n")


--- META_engineered_with_targets.csv ---
         Date        Open        High         Low       Close    Volume  \
0  2019-03-14  168.833765  170.216180  167.242504  169.241531  18037400   
1  2019-03-15  166.247972  166.665678  161.623333  165.074402  37135400   
2  2019-03-18  162.677544  163.005730  158.410942  159.594452  37524200   
3  2019-03-19  160.598926  162.926170  159.942538  160.688446  25611500   
4  2019-03-20  160.618833  165.213621  160.360257  164.537338  20211500   

       SMA_20      SMA_50      EMA_20      EMA_50  ...       lag_2  \
0  165.388666  156.839965  166.151734  158.366328  ...  170.981979   
1  165.489613  157.442658  166.049131  158.629390  ...  172.424072   
2  165.388666  158.014123  165.434399  158.667235  ...  169.241531   
3  165.352863  158.483945  164.982404  158.746498  ...  165.074402   
4  165.496076  159.028756  164.940017  158.973590  ...  159.594452   

        lag_3       lag_5      lag_10  roll_min_20  roll_max_20  roll_mean_20  \
0  17